# Inspect per-paddock time series

The `(paddock, time)` Zarr cube produced by `make_paddock_time_series`
holds the per-paddock median of every Sentinel-2 band and every
spectral index at every observation. This is the central data product
— phenology, plots, calendars, the PDF report all derive from it.

This notebook shows how to load it, slice it, compare paddocks /
indices / years, and apply smoothing.

> **Prerequisite.** A pipeline run for the query below. Edit
> `bbox` / `start` / `end` / `stub` to point at a run you've completed.


In [ ]:
from datetime import date
from PaddockTS.query import Query

query = Query(
    bbox=[148.36265, -33.52606, 148.38265, -33.50606],
    start=date(2024, 1, 1),
    end=date(2024, 12, 31),
    stub="stages_demo",
)


## 1. Load the cube


In [ ]:
import xarray as xr

ts_path = f"{query.tmp_dir}/sam_paddocks_timeseries.zarr"
ts = xr.open_zarr(ts_path, decode_coords="all")
print(ts)


In [ ]:
print("variables:", list(ts.data_vars))
print("paddocks :", ts.sizes["paddock"])
print("timesteps:", ts.sizes["time"])
print("time span:", str(ts.time.values[0])[:10], "→", str(ts.time.values[-1])[:10])


## 2. One paddock's NDVI trajectory


In [ ]:
import matplotlib.pyplot as plt

paddock_id = ts.paddock.values[0]
fig, ax = plt.subplots(figsize=(11, 4))
ts.NDVI.sel(paddock=paddock_id).plot(ax=ax, marker=".", color="green")
ax.set_title(f"Paddock {paddock_id} — NDVI")
ax.set_ylabel("NDVI")
plt.tight_layout()
plt.show()


## 3. Compare multiple paddocks


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for p in ts.paddock.values[:6]:
    ts.NDVI.sel(paddock=p).plot(ax=ax, label=str(p), marker=".")
ax.set_title("NDVI across the first six paddocks")
ax.set_ylabel("NDVI")
ax.legend(ncol=6, loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()


## 4. Compare multiple indices for one paddock


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for var, color in zip(["NDVI", "CFI", "NIRv"], ["green", "darkorange", "steelblue"]):
    ts[var].sel(paddock=paddock_id).plot(ax=ax, label=var, marker=".", color=color)
ax.set_title(f"Paddock {paddock_id} — vegetation indices")
ax.set_ylabel("index value")
ax.legend()
plt.tight_layout()
plt.show()


## 5. AOI-average NDVI

Average across all paddocks gives a quick AOI-level signal.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ts.NDVI.mean(dim="paddock").plot(ax=ax, marker=".", color="darkgreen")
ax.set_title("AOI-mean NDVI across all paddocks")
ax.set_ylabel("mean NDVI")
plt.tight_layout()
plt.show()


## 6. Smoothed series for phenology / trend analysis

Sentinel-2 has revisit gaps and cloud-mask drops, so the raw series is
irregular. `make_smoothed_paddock_time_series` resamples to a fixed
cadence, fills gaps with monotone PCHIP interpolation, and smooths with
a Savitzky-Golay filter.


In [ ]:
from PaddockTS.Phenology.make_smoothed_paddock_time_series import make_smoothed_paddock_time_series

smoothed = make_smoothed_paddock_time_series(
    query, ds_paddockTS=ts, days=10, window_length=7, polyorder=2,
)
print(smoothed)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ts.NDVI.sel(paddock=paddock_id).plot(
    ax=ax, marker="o", linestyle="", alpha=0.5, color="steelblue", label="raw")
smoothed.NDVI.sel(paddock=paddock_id).plot(
    ax=ax, color="red", linewidth=2, label="smoothed (10-day PCHIP + Savitzky-Golay)")
ax.set_title(f"Paddock {paddock_id} — raw vs smoothed NDVI")
ax.set_ylabel("NDVI")
ax.legend()
plt.tight_layout()
plt.show()


## See also

- [`05_inspect_phenology.ipynb`](05_inspect_phenology.ipynb) — derive seasonal metrics from this cube.
- [`04_inspect_paddocks.ipynb`](04_inspect_paddocks.ipynb) — join paddock geometry with time-series-derived metrics for thematic maps.
